| título | projeto | versão | data | autores | status |
| :--- | :--- | :--- | :--- | :--- | :--- |
| CRISP-DM — Fase 4: Modeling | Projeção da Taxa de Congestionamento — Justiça Estadual (GO) | 1.0 | 14-12-2025 | Júlio César e Lays de Freitas | Rascunho |


Esse Notebook contém a *Modelagem Preditiva (Baseline)*.

### BIBLIOTECAS

In [3]:
### BIBLIOTECAS (Adicionar estas linhas na seção de importações existente)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime as dt

from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import LabelEncoder
from dateutil.relativedelta import relativedelta

# Adicionar estas novas importações
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# Configurações visuais
plt.style.use('ggplot')
pd.set_option('display.max_columns', None)

### IMPORTAÇÃO E CONFIGURAÇÃO

In [4]:
# Carregamento dos dados processados na fase anterior
# Nota: Concatenamos treino e teste para refazer a divisão de forma TEMPORAL (passado vs futuro)

# Configurações visuais
plt.style.use('ggplot')
pd.set_option('display.max_columns', None)

try:
    df_train = pd.read_csv('datasets/train-processed.csv')
    df_test_split = pd.read_csv('datasets/test_split.csv')
    df_full = pd.concat([df_train, df_test_split], ignore_index=True)
except FileNotFoundError:
    # Caso não encontre os arquivos, usamos um dataframe simulado baseado na estrutura conhecida
    print("Arquivos não encontrados. Certifique-se de que a Fase 3 foi executada.")

# Garantir que a coluna de data é datetime
df_full['mes_ref'] = pd.to_datetime(df_full['mes_ref'])
#df_full['Taxa de Congestionamento_mes'] = df_full['Taxa de Congestionamento_mes (%)'] * 100  # Converter para porcentagem

# Ordenar por unidade e data
df_full = df_full.sort_values(by=['comarca', 'serventia', 'mes_ref'])

print(f"Total de registros carregados: {len(df_full)}")
print(f"Período dos dados: de {df_full['mes_ref'].min().date()} até {df_full['mes_ref'].max().date()}")

Total de registros carregados: 571917
Período dos dados: de 2022-01-01 até 2025-10-01


### REFINAMENTO DOS DADOS

In [5]:
# Vamos aplicar as regras de negócio para limpar a base antes do split temporal

# 1. Filtro de Tipo de Unidade: Remover CEJUSCs
# CEJUSCs costumam ter taxas de 0% ou 100% devido a mutirões
filtro_cejusc = ~df_full['serventia'].str.contains('CEJUSC|CENTRO JUDICIÁRIO', case=False, na=False)

# 2. Filtro de Volume (Tratamento de Zeros/Pequenas Amostras)
# Regra: Unidades com menos de 10 processos movimentados (Pendentes + Baixados) no mês são instáveis
# Se Pendentes=1 e Baixados=0 -> Taxa 100%. Se mês seguinte Pendentes=0 e Baixados=1 -> Taxa 0%.
df_full['volume_total'] = df_full['Pendentes_mes'] + df_full['Baixados_mes']
filtro_volume = df_full['volume_total'] >= 10

# Aplicação dos Filtros
df_refinado = df_full[filtro_cejusc & filtro_volume].copy()

# Estatísticas do Refinamento
total_original = len(df_full)
total_refinado = len(df_refinado)
removidos = total_original - total_refinado

print(f"Registros Originais: {total_original}")
print(f"Registros Após Refinamento: {total_refinado}")
print(f"Registros Removidos (Ruído): {removidos} ({removidos/total_original:.1%} da base)")

Registros Originais: 571917
Registros Após Refinamento: 397662
Registros Removidos (Ruído): 174255 (30.5% da base)


### FEATURE ENGINEERING E DIVISÃO TEMPORAL

In [6]:
print("Iniciando preparação para XGBoost...")

def criar_features_temporais(df_unidade, horizonte_previsao=1):
    """
    Cria features temporais para XGBoost a partir de uma série temporal
    """
    df = df_unidade.copy()
    
    # Garantir que está ordenado por tempo
    df = df.sort_values('mes_ref')
    
    # Features básicas de data
    df['ano'] = df['mes_ref'].dt.year
    df['mes'] = df['mes_ref'].dt.month
    df['trimestre'] = df['mes_ref'].dt.quarter
    
    # Features cíclicas para capturar sazonalidade
    df['mes_sin'] = np.sin(2 * np.pi * df['mes']/12)
    df['mes_cos'] = np.cos(2 * np.pi * df['mes']/12)
    df['trimestre_sin'] = np.sin(2 * np.pi * df['trimestre']/4)
    df['trimestre_cos'] = np.cos(2 * np.pi * df['trimestre']/4)
    
    # Variável alvo (taxa de congestionamento)
    df['target'] = df['Taxa de Congestionamento_mes (%)']
    
    # Features de lag (valores passados)
    for lag in [1, 2, 3, 4, 5, 6, 12]:  # 1-6 meses e 1 ano
        if lag < len(df):
            df[f'lag_{lag}'] = df['target'].shift(lag)
    
    # Médias móveis
    for window in [3, 6, 12]:
        if window < len(df):
            df[f'media_movel_{window}'] = df['target'].rolling(window=window, min_periods=1).mean().shift(1)
            df[f'std_movel_{window}'] = df['target'].rolling(window=window, min_periods=1).std().shift(1)
    
    # Features de tendência
    if len(df) > 1:
        df['tendencia'] = np.arange(len(df))
        df['tendencia_quad'] = df['tendencia'] ** 2
        df['tendencia_sqrt'] = np.sqrt(df['tendencia'])
    
    # Features de diferença
    if len(df) > 1:
        df['diff_1'] = df['target'].diff(1)
        df['diff_12'] = df['target'].diff(12) if len(df) > 12 else np.nan
    
    # Remover linhas com NaN (devido aos lags)
    df_clean = df.dropna().copy()
    
    if len(df_clean) == 0:
        return None
    
    # Definir variáveis de entrada (X) e saída (y)
    # Excluir colunas que não são features
    exclude_cols = ['comarca', 'serventia', 'mes_ref', 'target', 
                    'Taxa de Congestionamento_mes (%)', 'mes_ordinal',
                    'Distribuídos_mes', 'Baixados_mes', 'Pendentes_mes', 'volume_total']
    
    feature_cols = [col for col in df_clean.columns if col not in exclude_cols]
    
    # Garantir que temos features suficientes
    if len(feature_cols) == 0 or len(df_clean) < 10:
        return None
    
    # Separar em treino e "teste" (últimos 'horizonte_previsao' meses)
    if len(df_clean) > horizonte_previsao * 2:  # Pelo menos 2x o horizonte para treino
        split_idx = -horizonte_previsao
        X_train = df_clean[feature_cols].iloc[:split_idx]
        y_train = df_clean['target'].iloc[:split_idx]
        X_test = df_clean[feature_cols].iloc[split_idx:]
        y_test = df_clean['target'].iloc[split_idx:]
        
        return {
            'X_train': X_train,
            'y_train': y_train,
            'X_test': X_test,
            'y_test': y_test,
            'feature_cols': feature_cols,
            'df_full': df_clean,
            'ultima_data': df_clean['mes_ref'].iloc[-1]
        }
    
    return None

Iniciando preparação para XGBoost...


In [8]:
# Testar a função em algumas unidades
print("\nCriando features temporais para amostra de unidades...")


dados_xgboost = {}

# Identificar todas as combinações únicas de Comarca e Serventia
unidades = df_refinado[['comarca', 'serventia']].drop_duplicates()

print(f"Iniciando treinamento para {len(unidades)} unidades jurisdicionais...")

unidades_amostra_xgb = unidades.sample(min(100, len(unidades)))

for index, row in unidades_amostra_xgb.iterrows():
    comarca_atual = row['comarca']
    serventia_atual = row['serventia']
    
    # Filtrar dados da unidade
    mask = (df_refinado['comarca'] == comarca_atual) & (df_refinado['serventia'] == serventia_atual)
    dados_unidade = df_refinado[mask]
    
    if len(dados_unidade) >= 24:  # Mínimo 2 anos para features temporais
        features = criar_features_temporais(dados_unidade, horizonte_previsao=3)
        
        if features is not None:
            dados_xgboost[(comarca_atual, serventia_atual)] = features
    
    # Progresso
    if (index + 1) % 20 == 0:
        print(f"Processadas {index + 1} unidades...")

print(f"\nFeatures criadas para {len(dados_xgboost)} unidades")


Criando features temporais para amostra de unidades...
Iniciando treinamento para 513 unidades jurisdicionais...
Processadas 288200 unidades...
Processadas 398900 unidades...
Processadas 293500 unidades...
Processadas 147960 unidades...
Processadas 271400 unidades...
Processadas 307220 unidades...

Features criadas para 99 unidades


### TREINAMENTO DO MODELO

In [9]:
### IMPLEMENTAÇÃO DO MODELO XGBOOST

def treinar_xgboost_otimizado(X_train, y_train, cv_folds=5):
    """
    Treina modelo XGBoost com otimização de hiperparâmetros
    """
    # Definir espaço de busca de hiperparâmetros
    param_grid = {
        'n_estimators': [50, 100, 200], 
        'max_depth': [3, 5, 7],
        'learning_rate': [0.01, 0.05, 0.1],
        'subsample': [0.8, 0.9, 1.0],
        'colsample_bytree': [0.8, 0.9, 1.0]
    }
    
    # Time Series Cross Validation
    tscv = TimeSeriesSplit(n_splits=min(cv_folds, len(X_train)-1))
    
    melhor_score = np.inf
    melhor_modelo = None
    melhores_params = None
    
    # Busca em grade simplificada
    print("  Otimizando hiperparâmetros...", end="")
    
    for n_est in param_grid['n_estimators']:
        for max_d in param_grid['max_depth']:
            for lr in param_grid['learning_rate']:
                for subsample in param_grid['subsample']:
                    for colsample in param_grid['colsample_bytree']:
                        
                        scores_cv = []
                        
                        for train_idx, val_idx in tscv.split(X_train):
                            X_train_cv = X_train.iloc[train_idx]
                            y_train_cv = y_train.iloc[train_idx]
                            X_val_cv = X_train.iloc[val_idx]
                            y_val_cv = y_train.iloc[val_idx]
                            
                            # Treinar modelo
                            modelo = xgb.XGBRegressor(
                                n_estimators=n_est,
                                max_depth=max_d,
                                learning_rate=lr,
                                subsample=subsample,
                                colsample_bytree=colsample,
                                random_state=42,
                                n_jobs=-1,
                                objective='reg:squarederror'
                            )
                            
                            modelo.fit(X_train_cv, y_train_cv)
                            preds = modelo.predict(X_val_cv)
                            
                            # Clipar previsões
                            preds = np.clip(preds, 0, 100)
                            
                            # Calcular MAE
                            mae = mean_absolute_error(y_val_cv, preds)
                            scores_cv.append(mae)
                        
                        # Score médio no CV
                        score_medio = np.mean(scores_cv)
                        
                        if score_medio < melhor_score:
                            melhor_score = score_medio
                            melhores_params = {
                                'n_estimators': n_est,
                                'max_depth': max_d,
                                'learning_rate': lr,
                                'subsample': subsample,
                                'colsample_bytree': colsample
                            }
    
    print(f" Concluído! Melhor MAE CV: {melhor_score:.2f}")
    
    # Treinar modelo final com melhores parâmetros
    modelo_final = xgb.XGBRegressor(**melhores_params, random_state=42, n_jobs=-1)
    modelo_final.fit(X_train, y_train)
    
    return modelo_final, melhores_params, melhor_score

# Treinar modelos XGBoost
print("\nTreinando modelos XGBoost...")

modelos_xgboost_dict = {}
resultados_xgboost = []

for idx, (chave, dados) in enumerate(dados_xgboost.items()):
    comarca, serventia = chave
    
    print(f"\n[{idx+1}/{len(dados_xgboost)}] Treinando para {comarca}...")
    
    X_train = dados['X_train']
    y_train = dados['y_train']
    X_test = dados['X_test']
    y_test = dados['y_test']
    feature_cols = dados['feature_cols']
    
    # Treinar modelo
    modelo_xgb, params, score_cv = treinar_xgboost_otimizado(X_train, y_train)
    
    # Avaliar no conjunto de teste
    y_pred = modelo_xgb.predict(X_test)
    y_pred = np.clip(y_pred, 0, 100)
    
    mae_test = mean_absolute_error(y_test, y_pred)
    rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))
    
    # Importância das features
    importancia = modelo_xgb.feature_importances_
    features_importantes = sorted(zip(feature_cols, importancia), 
                                  key=lambda x: x[1], reverse=True)[:5]
    
    # Armazenar modelo
    modelos_xgboost_dict[chave] = {
        'modelo': modelo_xgb,
        'params': params,
        'feature_cols': feature_cols,
        'score_cv': score_cv,
        'mae_test': mae_test,
        'rmse_test': rmse_test,
        'features_importantes': features_importantes,
        'ultima_data': dados['ultima_data'],
        'df_full': dados['df_full']
    }
    
    resultados_xgboost.append({
        'comarca': comarca,
        'serventia': serventia,
        'n_observacoes': len(dados['df_full']),
        'n_treino': len(X_train),
        'n_teste': len(X_test),
        'mae_cv': score_cv,
        'mae_teste': mae_test,
        'rmse_teste': rmse_test,
        'melhores_params': params,
        'top_features': features_importantes
    })

print(f"\n✅ Modelos XGBoost treinados: {len(modelos_xgboost_dict)}")


Treinando modelos XGBoost...

[1/99] Treinando para ÁGUAS LINDAS DE GOIÁS...
  Otimizando hiperparâmetros... Concluído! Melhor MAE CV: 4.23

[2/99] Treinando para ANICUNS...
  Otimizando hiperparâmetros... Concluído! Melhor MAE CV: 4.04

[3/99] Treinando para PALMEIRAS DE GOIÁS...
  Otimizando hiperparâmetros... Concluído! Melhor MAE CV: 0.55

[4/99] Treinando para ALEXÂNIA...
  Otimizando hiperparâmetros... Concluído! Melhor MAE CV: 1.25

[5/99] Treinando para TRIBUNAL DE JUSTIÇA...
  Otimizando hiperparâmetros... Concluído! Melhor MAE CV: 7.80

[6/99] Treinando para GOIÂNIA...
  Otimizando hiperparâmetros... Concluído! Melhor MAE CV: 2.21

[7/99] Treinando para TRIBUNAL DE JUSTIÇA...
  Otimizando hiperparâmetros... Concluído! Melhor MAE CV: 8.46

[8/99] Treinando para GUAPÓ...
  Otimizando hiperparâmetros... Concluído! Melhor MAE CV: 0.64

[9/99] Treinando para GOIÂNIA...
  Otimizando hiperparâmetros... Concluído! Melhor MAE CV: 1.53

[10/99] Treinando para TRIBUNAL DE JUSTIÇA...
  

In [10]:
resultados_xgboost = pd.DataFrame(resultados_xgboost)
resultados_xgboost.to_csv('results/resultados_xgboost.csv', index=False)

### GERAÇÃO DE PREVISÕES FUTURAS

In [11]:
### PREVISÕES FUTURAS COM XGBOOST

def criar_features_futuras(df_historico, horizon, ultima_data):
    """
    Cria features para datas futuras baseadas no histórico
    """
    # Criar datas futuras
    datas_futuras = [ultima_data + relativedelta(months=i+1) for i in range(horizon)]
    
    # Começar com o último registro do histórico
    df_futuro = df_historico.tail(horizon * 2).copy()  # Pegar dados suficientes para calcular features
    
    features_futuras = []
    
    for i, data_futura in enumerate(datas_futuras):
        # Criar linha para data futura
        nova_linha = {}
        
        # Features de data
        nova_linha['ano'] = data_futura.year
        nova_linha['mes'] = data_futura.month
        nova_linha['trimestre'] = data_futura.quarter
        
        # Features cíclicas
        nova_linha['mes_sin'] = np.sin(2 * np.pi * nova_linha['mes']/12)
        nova_linha['mes_cos'] = np.cos(2 * np.pi * nova_linha['mes']/12)
        nova_linha['trimestre_sin'] = np.sin(2 * np.pi * nova_linha['trimestre']/4)
        nova_linha['trimestre_cos'] = np.cos(2 * np.pi * nova_linha['trimestre']/4)
        
        # Tendência
        nova_linha['tendencia'] = len(df_historico) + i
        nova_linha['tendencia_quad'] = nova_linha['tendencia'] ** 2
        nova_linha['tendencia_sqrt'] = np.sqrt(nova_linha['tendencia'])
        
        # Para features de lag, usar valores previstos anteriores
        # Isso requer previsões recursivas
        features_futuras.append(nova_linha)
    
    return datas_futuras, features_futuras



In [12]:
print("\nGerando previsões futuras com XGBoost...")

previsoes_xgboost_por_horizonte = {horizon: [] for horizon in [3, 6, 12]}

for idx, (chave, info) in enumerate(modelos_xgboost_dict.items()):
    comarca, serventia = chave
    modelo_xgb = info['modelo']
    feature_cols = info['feature_cols']
    df_full = info['df_full']
    ultima_data = info['ultima_data']
    
    print(f"[{idx+1}/{len(modelos_xgboost_dict)}] Prevendo para {comarca}...")
    
    for horizon in [3, 6, 12]:
        try:
            # Abordagem recursiva para previsões multi-step
            df_historico = df_full.copy()
            previsoes = []
            
            for passo in range(horizon):
                # Criar features para próximo passo
                datas_futuras, features_list = criar_features_futuras(df_historico, 1, ultima_data)
                
                if not features_list:
                    break
                    
                features_futuro = features_list[0]
                
                # Adicionar lags e médias móveis do histórico
                # Pegar últimos valores conhecidos
                for lag in [1, 2, 3, 4, 5, 6, 12]:
                    col_name = f'lag_{lag}'
                    if col_name in feature_cols:
                        if lag <= len(df_historico):
                            # Usar valor real ou previsto
                            if passo == 0:
                                features_futuro[col_name] = df_historico['target'].iloc[-lag]
                            else:
                                # Para lags maiores que o número de previsões já feitas
                                if lag > passo:
                                    features_futuro[col_name] = df_historico['target'].iloc[-lag + passo]
                                else:
                                    features_futuro[col_name] = previsoes[-lag]
                
                # Adicionar médias móveis
                for window in [3, 6, 12]:
                    for stat in ['media_movel', 'std_movel']:
                        col_name = f'{stat}_{window}'
                        if col_name in feature_cols:
                            # Calcular com dados disponíveis
                            dados_disponiveis = list(df_historico['target'].iloc[-window:])
                            if len(previsoes) > 0:
                                dados_disponiveis.extend(previsoes[-min(window, len(previsoes)):])
                            
                            if len(dados_disponiveis) >= 2:
                                if stat == 'media_movel':
                                    features_futuro[col_name] = np.mean(dados_disponiveis[-window:])
                                else:
                                    features_futuro[col_name] = np.std(dados_disponiveis[-window:]) if len(dados_disponiveis) > 1 else 0
                
                # Criar DataFrame para previsão
                df_pred = pd.DataFrame([features_futuro])
                
                # Garantir que temos todas as colunas necessárias
                missing_cols = set(feature_cols) - set(df_pred.columns)
                for col in missing_cols:
                    df_pred[col] = 0
                
                df_pred = df_pred[feature_cols]
                
                # Fazer previsão
                pred = modelo_xgb.predict(df_pred)[0]
                pred = max(0, min(100, pred))
                previsoes.append(pred)
                
                # Atualizar "histórico" com previsão
                nova_linha = df_historico.iloc[-1:].copy()
                nova_linha['target'] = pred
                # Ajustar data (aproximação)
                nova_data = ultima_data + relativedelta(months=passo+1)
                nova_linha['mes_ref'] = nova_data
                df_historico = pd.concat([df_historico, nova_linha], ignore_index=True)
            
            # Armazenar previsões
            for i, (data_futura, pred) in enumerate(zip(
                [ultima_data + relativedelta(months=j+1) for j in range(horizon)],
                previsoes
            )):
                previsoes_xgboost_por_horizonte[horizon].append({
                    'comarca': comarca,
                    'serventia': serventia,
                    'data_futura': data_futura,
                    'horizonte_meses': horizon,
                    'taxa_prevista_xgboost': round(pred, 2),
                    'taxa_prevista_linear': None,  # Será preenchido depois
                    'mes_no_horizonte': i + 1
                })
                
        except Exception as e:
            print(f"  Erro na previsão XGBoost para {comarca}: {str(e)[:100]}...")
            continue


Gerando previsões futuras com XGBoost...
[1/99] Prevendo para ÁGUAS LINDAS DE GOIÁS...
[2/99] Prevendo para ANICUNS...
[3/99] Prevendo para PALMEIRAS DE GOIÁS...
[4/99] Prevendo para ALEXÂNIA...
[5/99] Prevendo para TRIBUNAL DE JUSTIÇA...
[6/99] Prevendo para GOIÂNIA...
[7/99] Prevendo para TRIBUNAL DE JUSTIÇA...
[8/99] Prevendo para GUAPÓ...
[9/99] Prevendo para GOIÂNIA...
[10/99] Prevendo para TRIBUNAL DE JUSTIÇA...
[11/99] Prevendo para GOIÂNIA...
[12/99] Prevendo para ANÁPOLIS...
[13/99] Prevendo para CIDADE OCIDENTAL...
[14/99] Prevendo para ITAPURANGA...
[15/99] Prevendo para TRIBUNAL DE JUSTIÇA...
[16/99] Prevendo para TRIBUNAL DE JUSTIÇA...
[17/99] Prevendo para GOIÂNIA...
[18/99] Prevendo para JOVIÂNIA...
[19/99] Prevendo para BOM JESUS DE GOIÁS...
[20/99] Prevendo para APARECIDA DE GOIÂNIA...
[21/99] Prevendo para GOIÂNIA...
[22/99] Prevendo para INHUMAS...
[23/99] Prevendo para LUZIÂNIA...
[24/99] Prevendo para ÁGUAS LINDAS DE GOIÁS...
[25/99] Prevendo para GOIÂNIA...
[26/9

In [13]:
# Converter para DataFrames
dfs_xgboost = {}
for horizon in [3, 6, 12]:
    if previsoes_xgboost_por_horizonte[horizon]:
        dfs_xgboost[horizon] = pd.DataFrame(previsoes_xgboost_por_horizonte[horizon])
    else:
        dfs_xgboost[horizon] = pd.DataFrame()

print("\nPrevisões XGBoost geradas:")
for horizon in [3, 6, 12]:
    print(f"  Horizonte {horizon} meses: {len(dfs_xgboost[horizon])} previsões")


Previsões XGBoost geradas:
  Horizonte 3 meses: 297 previsões
  Horizonte 6 meses: 594 previsões
  Horizonte 12 meses: 1188 previsões


In [24]:
dfs_xgboost[3]['comarca'].unique()

array(['ÁGUAS LINDAS DE GOIÁS', 'ANICUNS', 'PALMEIRAS DE GOIÁS',
       'ALEXÂNIA', 'TRIBUNAL DE JUSTIÇA', 'GOIÂNIA', 'GUAPÓ', 'ANÁPOLIS',
       'CIDADE OCIDENTAL', 'ITAPURANGA', 'JOVIÂNIA', 'BOM JESUS DE GOIÁS',
       'APARECIDA DE GOIÂNIA', 'INHUMAS', 'LUZIÂNIA', 'PLANALTINA',
       'TAQUARAL DE GOIÁS', 'PARANAIGUARA', 'ARAGARÇAS', 'ARAÇU', 'CERES',
       'MAURILÂNDIA', 'HIDROLÂNDIA', 'GOIANÉSIA',
       'SANTO ANTÔNIO DO DESCOBERTO', 'JUSSARA', 'CALDAS NOVAS',
       'MOZARLÂNDIA', 'FAZENDA NOVA', 'JATAÍ', 'JANDAIA', 'ORIZONA',
       'GOIANDIRA', 'SENADOR CANEDO', 'PORANGATU', 'QUIRINÓPOLIS',
       'ACREÚNA', 'GOIANIRA', 'RIO VERDE', 'ITUMBIARA', 'NOVO GAMA',
       'NERÓPOLIS', 'ITABERAÍ', 'CATALÃO', 'SÃO LUÍS DE MONTES BELOS',
       'JARAGUÁ'], dtype=object)

In [28]:
dfs_xgboost[3][dfs_xgboost[3]['comarca'] == 'GOIÂNIA']

,comarca,serventia,data_futura,horizonte_meses,taxa_prevista_xgboost,taxa_prevista_linear,mes_no_horizonte
15,GOIÂNIA,6ª Vara Criminal dos crimes punidos com reclus...,2025-11-01,3,100.000000,None,1
16,GOIÂNIA,6ª Vara Criminal dos crimes punidos com reclus...,2025-12-01,3,100.000000,None,2
17,GOIÂNIA,6ª Vara Criminal dos crimes punidos com reclus...,2026-01-01,3,100.000000,None,3
24,GOIÂNIA,2ª Vara de Família,2025-11-01,3,100.000000,None,1
25,GOIÂNIA,2ª Vara de Família,2025-12-01,3,100.000000,None,2
26,GOIÂNIA,2ª Vara de Família,2026-01-01,3,100.000000,None,3
30,GOIÂNIA,7ª Vara Criminal dos crimes punidos com reclus...,2025-11-01,3,100.000000,None,1
31,GOIÂNIA,7ª Vara Criminal dos crimes punidos com reclus...,2025-12-01,3,100.000000,None,2
32,GOIÂNIA,7ª Vara Criminal dos crimes punidos com reclus...,2026-01-01,3,100.000000,None,3
48,GOIÂNIA,3º Juizado Especial Criminal,2025-11-01,3,61.419998,None,1
